In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_rel
import ast
import os

# ========== CONFIGURATION ==========
MODEL_CONFIGS = {
    'HistGen': {
        'base_dir': '/home/woody/iwi5/iwi5204h/HistGen/Data/TestResult_HistGen',
        'seeds': {
            'seed43': {'seed_folder': '17_seed43', 'best_folder': 'Best_31'},
            'seed44': {'seed_folder': '17_seed44', 'best_folder': 'Best_35'},
            'seed45': {'seed_folder': '17_seed45', 'best_folder': 'Best_18'},
            'seed46': {'seed_folder': '17_seed46', 'best_folder': 'Best_35'},
            'seed456789': {'seed_folder': '17', 'best_folder': 'Best_39'}
        }
    },
    'TITAN': {
        'base_dir': '/home/woody/iwi5/iwi5204h/HistGen4TITAN/Data/TestResults',
        'seeds': {
            'seed43': {'seed_folder': '5_seed43', 'best_folder': 'Best17'},
            'seed44': {'seed_folder': '5_seed44', 'best_folder': 'Best30'},
            'seed45': {'seed_folder': '5_seed45', 'best_folder': 'Best40'},
            'seed46': {'seed_folder': '5_seed46', 'best_folder': 'Best37'},
            'seed456789': {'seed_folder': '5', 'best_folder': 'Best33'}
        }
    },
    'UNI1': {
        'base_dir': '/home/woody/iwi5/iwi5204h/HistGen/Data/TestResult_UNI1',
        'seeds': {
            'seed43': {'seed_folder': '1_seed43', 'best_folder': 'Best_15'},
            'seed44': {'seed_folder': '1_seed44', 'best_folder': 'Best_27'},
            'seed45': {'seed_folder': '1_seed45', 'best_folder': 'Best_38'},
            'seed46': {'seed_folder': '1_seed46', 'best_folder': 'Best_40'},
            'seed456789': {'seed_folder': '1', 'best_folder': 'Best_38'}
        }
    },
    'UNI2_4': {
        'base_dir': '/home/woody/iwi5/iwi5204h/HistGen/Data/TestResult_UNI2',
        'seeds': {
            'seed43': {'seed_folder': '4_seed43', 'best_folder': 'Best_34'},
            'seed44': {'seed_folder': '4_seed44', 'best_folder': 'Best_16'},
            'seed45': {'seed_folder': '4_seed45', 'best_folder': 'Best_38'},
            'seed46': {'seed_folder': '4_seed46', 'best_folder': 'Best_25'},
            'seed456789': {'seed_folder': '4', 'best_folder': 'Best_27'}
        }
    }
}

METRICS = ['test_BLEU_1', 'test_BLEU_2', 'test_BLEU_3', 'test_BLEU_4', 
           'test_METEOR', 'test_ROUGE_L']

# Bootstrap configuration
N_BOOTSTRAP = 10000  # Number of bootstrap iterations
CI_LEVEL = 0.95      # Confidence level (95%)

# ========== HELPER FUNCTIONS ==========
def load_scores_from_txt(txt_path):
    """Load scores from results.txt"""
    try:
        with open(txt_path, 'r') as f:
            content = f.read().strip()
        scores_dict = ast.literal_eval(content)
        scores = {}
        for metric in METRICS:
            if metric in scores_dict:
                scores[metric] = float(scores_dict[metric])
            else:
                scores[metric] = None
        return scores
    except Exception as e:
        print(f"    ERROR loading {txt_path}: {e}")
        return None

def load_scores_from_csv(csv_path):
    """Load scores from results.csv"""
    try:
        df = pd.read_csv(csv_path)
        scores = {}
        for metric in METRICS:
            if metric in df.columns:
                scores[metric] = float(df[metric].iloc[0])
            else:
                scores[metric] = None
        return scores
    except Exception as e:
        print(f"    ERROR loading {csv_path}: {e}")
        return None

def load_scores_for_model_seed(base_dir, seed_folder, best_folder):
    """Load scores from the specified path"""
    folder_path = os.path.join(base_dir, seed_folder, best_folder)
    
    # Try .txt first
    txt_path = os.path.join(folder_path, 'results.txt')
    if os.path.exists(txt_path):
        return load_scores_from_txt(txt_path)
    
    # Try .csv if .txt not found
    csv_path = os.path.join(folder_path, 'results.csv')
    if os.path.exists(csv_path):
        return load_scores_from_csv(csv_path)
    
    print(f"    WARNING: Neither results.txt nor results.csv found in {folder_path}")
    return None

def load_all_scores(model_configs):
    """Load all scores for all models and seeds"""
    all_scores = {}
    
    for model, config in model_configs.items():
        all_scores[model] = {}
        
        for seed_name, seed_config in config['seeds'].items():
            seed_folder = seed_config['seed_folder']
            best_folder = seed_config['best_folder']
            
            scores = load_scores_for_model_seed(
                config['base_dir'], 
                seed_folder, 
                best_folder
            )
            
            if scores:
                all_scores[model][seed_name] = scores
    
    return all_scores

def cohens_d(group1, group2):
    """Calculate Cohen's d effect size for paired samples"""
    diffs = np.array(group1) - np.array(group2)
    return np.mean(diffs) / np.std(diffs, ddof=1)

def interpret_cohens_d(d):
    """Interpret Cohen's d effect size"""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"

def bootstrap_ci(group1, group2, n_bootstrap=10000, ci_level=0.95):
    """
    Calculate bootstrap confidence interval for mean difference
    
    Parameters:
    -----------
    group1, group2 : array-like
        Paired samples (must have same length)
    n_bootstrap : int
        Number of bootstrap iterations (default: 10000)
    ci_level : float
        Confidence level, e.g., 0.95 for 95% CI (default: 0.95)
    
    Returns:
    --------
    mean_diff : float
        Observed mean difference
    ci_lower : float
        Lower bound of confidence interval
    ci_upper : float
        Upper bound of confidence interval
    """
    # Calculate differences
    diffs = np.array(group1) - np.array(group2)
    n = len(diffs)
    observed_mean = np.mean(diffs)
    
    # Bootstrap resampling
    bootstrap_means = np.zeros(n_bootstrap)
    for i in range(n_bootstrap):
        # Resample with replacement
        resample_indices = np.random.choice(n, size=n, replace=True)
        resample_diffs = diffs[resample_indices]
        bootstrap_means[i] = np.mean(resample_diffs)
    
    # Calculate percentile-based confidence interval
    alpha = 1 - ci_level
    ci_lower = np.percentile(bootstrap_means, 100 * alpha / 2)
    ci_upper = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    
    return observed_mean, ci_lower, ci_upper

# ========== PAIRED T-TEST ANALYSIS WITH BOOTSTRAP CI ==========
def perform_paired_ttest_analysis(all_scores, baseline_model='HistGen'):
    """
    Perform paired t-test comparing each model to baseline (HistGen) 
    for each metric across seeds, with bootstrap confidence intervals
    """
    print("="*70)
    print(f"PAIRED T-TEST ANALYSIS WITH BOOTSTRAP CI (Baseline: {baseline_model})")
    print("="*70)
    
    # Get list of seeds
    seeds = sorted(all_scores[baseline_model].keys())
    print(f"\nSeeds: {seeds}")
    print(f"Number of seeds: {len(seeds)}")
    print(f"Bootstrap iterations: {N_BOOTSTRAP}")
    print(f"Confidence level: {CI_LEVEL*100}%\n")
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Get comparison models (all except baseline)
    comparison_models = [m for m in sorted(all_scores.keys()) if m != baseline_model]
    
    results = []
    
    for metric in METRICS:
        print(f"\n{'='*70}")
        print(f"Metric: {metric}")
        print(f"{'='*70}")
        
        # Load baseline scores for this metric across all seeds
        baseline_scores = []
        for seed in seeds:
            if seed in all_scores[baseline_model]:
                score = all_scores[baseline_model][seed].get(metric)
                if score is not None:
                    baseline_scores.append(score)
                else:
                    print(f"  WARNING: Missing {metric} for {baseline_model}/{seed}")
                    baseline_scores.append(np.nan)
            else:
                print(f"  WARNING: Missing data for {baseline_model}/{seed}")
                baseline_scores.append(np.nan)
        
        baseline_scores = np.array(baseline_scores)
        
        # Compare each model to baseline
        for model in comparison_models:
            print(f"\n--- {model} vs {baseline_model} ---")
            
            # Load model scores for this metric across all seeds
            model_scores = []
            for seed in seeds:
                if seed in all_scores[model]:
                    score = all_scores[model][seed].get(metric)
                    if score is not None:
                        model_scores.append(score)
                    else:
                        print(f"  WARNING: Missing {metric} for {model}/{seed}")
                        model_scores.append(np.nan)
                else:
                    print(f"  WARNING: Missing data for {model}/{seed}")
                    model_scores.append(np.nan)
            
            model_scores = np.array(model_scores)
            
            # Check for missing data
            valid_mask = ~(np.isnan(baseline_scores) | np.isnan(model_scores))
            valid_baseline = baseline_scores[valid_mask]
            valid_model = model_scores[valid_mask]
            
            if len(valid_baseline) < 2:
                print(f"  ERROR: Not enough valid pairs (n={len(valid_baseline)})")
                continue
            
            # Calculate statistics
            baseline_mean = np.mean(valid_baseline)
            baseline_std = np.std(valid_baseline, ddof=1)
            model_mean = np.mean(valid_model)
            model_std = np.std(valid_model, ddof=1)
            mean_diff = model_mean - baseline_mean
            
            print(f"  {baseline_model}: {baseline_mean:.4f} ± {baseline_std:.4f}")
            print(f"  {model}: {model_mean:.4f} ± {model_std:.4f}")
            print(f"  Mean difference: {mean_diff:.4f}")
            
            # Paired t-test
            try:
                t_stat, p_value = ttest_rel(valid_model, valid_baseline)
                print(f"  Paired t-test: t={t_stat:.4f}, p={p_value:.4g}")
            except Exception as e:
                print(f"  ERROR in t-test: {e}")
                t_stat, p_value = np.nan, np.nan
            
            # Cohen's d (for paired samples)
            try:
                d = cohens_d(valid_model, valid_baseline)
                d_interpretation = interpret_cohens_d(d)
                print(f"  Cohen's d: {d:.4f} ({d_interpretation} effect)")
            except Exception as e:
                print(f"  ERROR calculating Cohen's d: {e}")
                d, d_interpretation = np.nan, "N/A"
            
            # Bootstrap confidence interval
            try:
                _, ci_lower, ci_upper = bootstrap_ci(
                    valid_model, valid_baseline, 
                    n_bootstrap=N_BOOTSTRAP, 
                    ci_level=CI_LEVEL
                )
                ci_excludes_zero = (ci_lower > 0) or (ci_upper < 0)
                print(f"  Bootstrap {int(CI_LEVEL*100)}% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
                print(f"  CI excludes zero: {ci_excludes_zero}")
            except Exception as e:
                print(f"  ERROR in bootstrap: {e}")
                ci_lower, ci_upper = np.nan, np.nan
                ci_excludes_zero = False
            
            # Significance
            if not np.isnan(p_value):
                if p_value < 0.001:
                    significance_marker = "***"
                elif p_value < 0.01:
                    significance_marker = "**"
                elif p_value < 0.05:
                    significance_marker = "*"
                else:
                    significance_marker = "ns"
                is_significant = p_value < 0.05
            else:
                significance_marker = "N/A"
                is_significant = False
            
            print(f"  Significant: {is_significant} ({significance_marker})")
            
            # Store results
            results.append({
                'Metric': metric,
                'Comparison': f'{model} vs {baseline_model}',
                'Model': model,
                'Baseline_Mean': baseline_mean,
                'Baseline_Std': baseline_std,
                'Model_Mean': model_mean,
                'Model_Std': model_std,
                'Mean_Diff': mean_diff,
                'n_pairs': len(valid_baseline),
                't_statistic': t_stat,
                'p_value': p_value,
                'Cohens_d': d,
                'Effect_Size': d_interpretation,
                'Boot_CI_Lower': ci_lower,
                'Boot_CI_Upper': ci_upper,
                'CI_Excludes_Zero': ci_excludes_zero,
                'Significant': is_significant,
                'Significance': significance_marker
            })
    
    return pd.DataFrame(results)

def save_results(results_df, output_prefix='paired_ttest'):
    """Save results in multiple formats"""
    # Save detailed results
    detailed_output = f'{output_prefix}_detailed.csv'
    results_df.to_csv(detailed_output, index=False)
    print(f"\nDetailed results saved to: {detailed_output}")
    
    # Create summary table (grouped by metric)
    print("\n" + "="*70)
    print("SUMMARY TABLE (Grouped by Metric)")
    print("="*70)
    
    for metric in results_df['Metric'].unique():
        metric_df = results_df[results_df['Metric'] == metric]
        print(f"\n{metric}:")
        display_cols = ['Model', 'Mean_Diff', 'p_value', 'Cohens_d', 
                       'Boot_CI_Lower', 'Boot_CI_Upper', 'CI_Excludes_Zero', 'Significance']
        print(metric_df[display_cols].to_string(index=False))
    
    # Create formatted summary
    summary_rows = []
    for _, row in results_df.iterrows():
        summary_rows.append({
            'Metric': row['Metric'],
            'Model': row['Model'],
            'Baseline': f"{row['Baseline_Mean']:.4f} ± {row['Baseline_Std']:.4f}",
            'Model_Score': f"{row['Model_Mean']:.4f} ± {row['Model_Std']:.4f}",
            'Diff': f"{row['Mean_Diff']:.4f}",
            'p-value': f"{row['p_value']:.4g}" if not np.isnan(row['p_value']) else "N/A",
            "Cohen's d": f"{row['Cohens_d']:.4f}" if not np.isnan(row['Cohens_d']) else "N/A",
            f'{int(CI_LEVEL*100)}% CI': f"[{row['Boot_CI_Lower']:.4f}, {row['Boot_CI_Upper']:.4f}]" if not np.isnan(row['Boot_CI_Lower']) else "N/A",
            'CI_Sig': '✓' if row['CI_Excludes_Zero'] else '✗',
            'Sig': row['Significance']
        })
    
    summary_df = pd.DataFrame(summary_rows)
    summary_output = f'{output_prefix}_summary.csv'
    summary_df.to_csv(summary_output, index=False)
    print(f"\nFormatted summary saved to: {summary_output}")
    
    return summary_df

# ========== MAIN ==========
def main():
    """Main analysis function"""
    print("="*70)
    print("LOADING SCORES FOR PAIRED T-TEST ANALYSIS WITH BOOTSTRAP CI")
    print("="*70)
    
    # Load all scores
    all_scores = load_all_scores(MODEL_CONFIGS)
    
    # Check what was loaded
    print(f"\nLoaded data for {len(all_scores)} models:")
    for model in sorted(all_scores.keys()):
        n_seeds = len(all_scores[model])
        print(f"  {model}: {n_seeds} seeds")
    
    # Perform paired t-test analysis with bootstrap CI
    results_df = perform_paired_ttest_analysis(all_scores, baseline_model='HistGen')
    
    # Save results
    summary_df = save_results(results_df, output_prefix='corpus_paired_ttest_with_bootstrap')
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE")
    print("="*70)
    
    return results_df, summary_df

# ========== RUN ==========
if __name__ == "__main__":
    results_df, summary_df = main()


LOADING SCORES FOR PAIRED T-TEST ANALYSIS WITH BOOTSTRAP CI

Loaded data for 4 models:
  HistGen: 5 seeds
  TITAN: 5 seeds
  UNI1: 5 seeds
  UNI2_4: 5 seeds
PAIRED T-TEST ANALYSIS WITH BOOTSTRAP CI (Baseline: HistGen)

Seeds: ['seed43', 'seed44', 'seed45', 'seed456789', 'seed46']
Number of seeds: 5
Bootstrap iterations: 10000
Confidence level: 95.0%


Metric: test_BLEU_1

--- TITAN vs HistGen ---
  HistGen: 0.7091 ± 0.0292
  TITAN: 0.7394 ± 0.0426
  Mean difference: 0.0303
  Paired t-test: t=1.0654, p=0.3467
  Cohen's d: 0.4765 (small effect)
  Bootstrap 95% CI: [-0.0260, 0.0717]
  CI excludes zero: False
  Significant: False (ns)

--- UNI1 vs HistGen ---
  HistGen: 0.7091 ± 0.0292
  UNI1: 0.7207 ± 0.0320
  Mean difference: 0.0116
  Paired t-test: t=0.5220, p=0.6292
  Cohen's d: 0.2335 (small effect)
  Bootstrap 95% CI: [-0.0258, 0.0512]
  CI excludes zero: False
  Significant: False (ns)

--- UNI2_4 vs HistGen ---
  HistGen: 0.7091 ± 0.0292
  UNI2_4: 0.7402 ± 0.0276
  Mean difference: